# Histopathologic Cancer Detection — runner
Thin orchestration for **Colab** or **Kaggle**. Heavy logic lives in `src/`.


## 0. Environment detect

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists('/kaggle/input')
print('Kaggle' if ON_KAGGLE else 'Colab/local')

# `kaggle kernels push` uploads only THIS notebook, so pull the code (public repo).
REPO_URL = 'https://github.com/aRealGem/histopath-cancer-detection'
REPO_DIR = ('/kaggle/working/histopath-cancer-detection' if ON_KAGGLE
            else 'histopath-cancer-detection')
if not os.path.exists(os.path.join('src', 'train.py')):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

## 1. Deps (safe on both — TF stays untouched)

In [ ]:
!pip -q install opencv-python-headless pyyaml scikit-learn

## 2. Data
- **Kaggle:** add the competition dataset via the sidebar; it mounts at `/kaggle/input/histopathologic-cancer-detection`. Nothing to download.
- **Colab:** upload `kaggle.json`, then run the download cell.


In [ ]:
if not ON_KAGGLE:
    from pathlib import Path
    Path.home().joinpath('.kaggle').mkdir(exist_ok=True)
    # from google.colab import files; files.upload()  # -> kaggle.json
    # !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !bash scripts/download_data.sh ./data
    # then edit configs/baseline.yaml -> data.root: ./data


## 3. Train -> Evaluate -> Predict

In [ ]:
# smoke.yaml validates the whole pipeline in minutes and writes a real submission.csv.
# For the full run, set HP_CONFIG=configs/baseline.yaml (or edit below) and re-run.
CFG = os.environ.get('HP_CONFIG', 'configs/smoke.yaml')
print('using config:', CFG)
!python -m src.train    --config {CFG}
!python -m src.evaluate --config {CFG}
!python -m src.predict  --config {CFG}

## 4. Inspect ROC + submission

In [ ]:
from IPython.display import Image, display
import pandas as pd, os
if os.path.exists('artifacts/roc_val.png'): display(Image('artifacts/roc_val.png'))
print(pd.read_csv('artifacts/submission.csv').head())

## 5. Submit
Kaggle: use the **Submit** button, or:
```
!kaggle competitions submit -c histopathologic-cancer-detection \
  -f artifacts/submission.csv -m 'MobileNetV3 baseline'
```
Then screenshot the leaderboard for the R09.1 done-criteria.